In [1]:
import macro_simulation
import numpy as np
import json

# Create Westbound Road

In [ ]:
network_id = "i24_westbound"
road_id = "2"
road_length = 1500.0
longitudinal_step = 50.0
lane_width = 3.6576
lane_count = 4
road_width = lane_width * lane_count
starting_x = 200.0
starting_y = 0.0
ending_x = 0.0

road_left_polyline = [
    (starting_x, starting_y),
    (ending_x, ((road_length ** 2) - (starting_x ** 2)) ** 0.5)
]

direction_vector = ((road_left_polyline[1][0] - road_left_polyline[0][0]) / road_length, (road_left_polyline[1][1] - road_left_polyline[0][1]) / road_length)
norm_vector = (direction_vector[1], -direction_vector[0]) # Goes to the right side of the road

road_right_polyline = [
    (road_left_polyline[0][0] + (road_width * norm_vector[0]), road_left_polyline[0][1] + (road_width * norm_vector[1])),
    (road_left_polyline[1][0] + (road_width * norm_vector[0]), road_left_polyline[1][1] + (road_width * norm_vector[1]))
]

road_lane_data = {
    -1: {
        "width": lane_width,
        "lateral_position": 0.0
    },
    -2: {
        "width": lane_width,
        "lateral_position": -lane_width
    },
    -3: {
        "width": lane_width,
        "lateral_position": -(lane_width*2.0)
    },
    -4: {
        "width": lane_width,
        "lateral_position": -(lane_width*3.0)
    }
}

longitudinal_steps = np.arange(0.0, road_length, longitudinal_step).tolist()
cells = {}
for i, step in enumerate(longitudinal_steps):
    for lane in road_lane_data:
        cell_id = f"road_{road_id}_cell_{lane}_step_{i}"
        start_s = step
        end_s = step + longitudinal_step
        density = 0
        inflow_connections = []
        outflow_connections = []
        if (i > 0):
            inflow_connections.append((road_id, f"road_{road_id}_cell_{lane}_step_{i - 1}"))
        if (i < (len(longitudinal_steps) - 1)):
            outflow_connections.append((road_id, f"road_{road_id}_cell_{lane}_step_{i + 1}"))
        cell = macro_simulation.Cell(road_id=road_id, cell_id=cell_id, lane=lane, start_s=start_s, end_s=end_s, density=density, inflow_connections=inflow_connections, outflow_connections=outflow_connections)
        cells[cell_id] = cell

road = macro_simulation.Road(road_id=road_id, left_polyline=road_left_polyline, right_polyline=road_right_polyline, lane_data=road_lane_data, cells=cells)
network = macro_simulation.Network(network_id=network_id, roads={road_id: road})

In [ ]:
network

In [ ]:
network.validate()

In [ ]:
network_dict = network.to_dict()

In [ ]:
network_dict

In [ ]:
with open("i24_westbound_network.json", "w+") as json_file:
    json.dump(network_dict, json_file, indent=4)

In [ ]:
network_reloaded = macro_simulation.Network.from_json("i24_westbound_network.json")

In [ ]:
network_reloaded

In [ ]:
network_reloaded.plot_network()

In [ ]:
network_reloaded.roads["2"].lane_data

# West and Eastbound Networks Combined

In [2]:
network = macro_simulation.I24WestAndEastNetwork()
network.create_network()
network.network.plot_network()